# 05 — Onset models: forecasting floods that have not started

**The problem this notebook solves.** The general model reports around 55% recall at
15 cm, 1 hour. Decompose it:

| Population | Positive rows | Model recall |
|---|---|---|
| Already flooded | ~430 | ~100% *(persistence gets these for free)* |
| Genuinely dry — true onsets | ~413 | **~9%** |

The headline is carried almost entirely by the easy half. On the half that requires actual
forecasting, the model catches roughly one in eleven.

**Why it happens.** `fl_depth_now` accounts for about 72% of the model's gain. Given a
feature that answers most of the question, gradient boosting will use it — that is not a
flaw, it is the algorithm working. But the result is a monitoring system wearing a
forecasting label.

**The fix, which is almost embarrassingly simple.** Train on onset rows only — rows where
the station is currently below the threshold. With the shortcut removed, the model has no
choice but to learn precursors: rainfall, rainfall intensification, canal rise.

**The result.**

| Horizon | Onset recall, general model | Onset recall, onset model |
|---|---|---|
| 1 h | 9% | **63%** |
| 3 h | — | 12% |
| 6 h | — | 5% *(too close to noise — not deployed)* |

**The catch, stated up front.** Precision is about 1%. The base rate is 0.01%, so that is a
30-60x lift and genuinely informative — but it means most notices do not lead to flooding.
So onset output raises a **Watch** and never an Advisory or a Warning. That is standard
meteorological practice: a watch says the ingredients are present, a warning says it is
happening.

In [1]:
import sys, pathlib
# Make the shared library importable no matter where Jupyter was started from.
ROOT = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
            if (p / "config" / "config.yaml").is_file())
sys.path.insert(0, str(ROOT / "src"))

import numpy as np
import pandas as pd
pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 60)

from bkkflood import CFG, PATHS
print("project root:", ROOT)
print("config version:", CFG["project"]["version"])
import json
from bkkflood.modeling import train_classifier, choose_operating_point, save_model, feature_importance
from bkkflood.metrics import classification_metrics, pick_threshold, event_scores
from bkkflood.splits import rolling_origin_folds, split_frame
from bkkflood.features import feature_columns, load_training_years

PRIMARY = CFG["flood_event"]["primary_tier_cm"]
ONSET_HORIZONS = CFG["alerting"]["onset_horizons_h"]
print(f"Primary tier {PRIMARY} cm; onset models deployed at {ONSET_HORIZONS}h")

project root: /Users/pritimmondal/Projects/bkk-flood-forecast
config version: 2.0.0


Primary tier 15 cm; onset models deployed at [1, 3]h


In [2]:
# Load per fold, not all at once — see the memory note in notebook 04.
from bkkflood.data import load_years, feature_list, available_years

YEARS = available_years()
FEATURES = feature_list()
ONSET_COL = f"is_onset_ge{PRIMARY}"
print(f"years built: {YEARS}")
print(f"{len(FEATURES)} features; onset column: {ONSET_COL}")

years built: [2019, 2020, 2021, 2022, 2023, 2024, 2025]
50 features; onset column: is_onset_ge15


## 1. Isolate the onset rows

One line, and it is the whole idea: keep only rows where the station is currently below
the threshold.

In [3]:
# How rare are onset positives? Counted year by year so nothing large stays
# resident. `class_imbalance.csv` from notebook 02 has the same numbers; this
# re-derives them on the actual feature tables as a consistency check.
label_cols = [f"y_ge{PRIMARY}_{h}h" for h in CFG["forecast"]["horizons_h"]]

totals = {h: {"rows": 0, "pos": 0} for h in CFG["forecast"]["horizons_h"]}
onset_rows = all_rows = 0

for year in YEARS:
    df = load_years([year], features=["fl_depth_now"], labels=label_cols,
                    extra=[ONSET_COL], verbose=False)
    if df.empty:
        continue
    all_rows += len(df)
    onset = df[df[ONSET_COL] == 1]
    onset_rows += len(onset)
    for h in CFG["forecast"]["horizons_h"]:
        col = f"y_ge{PRIMARY}_{h}h"
        if col in onset.columns:
            totals[h]["rows"] += len(onset)
            totals[h]["pos"] += int(onset[col].sum())
    del df, onset

print(f"All rows:   {all_rows:,}")
print(f"Onset rows: {onset_rows:,}  ({onset_rows / max(all_rows, 1):.1%})")
print()
for h, t in totals.items():
    print(f"  {h}h: {t['pos']:,} onset positives "
          f"(1 in {t['rows'] // max(t['pos'], 1):,} rows)")

All rows:   24,050,653
Onset rows: 24,045,652  (100.0%)

  1h: 3,427 onset positives (1 in 7,016 rows)
  3h: 10,044 onset positives (1 in 2,394 rows)
  6h: 19,868 onset positives (1 in 1,210 rows)


**Look at that base rate.** Roughly one in ten thousand. This is a needle-in-a-haystack
problem, and it is worth being clear about what "good" can mean here: at a base rate of
0.01%, precision of 1% is a hundred-fold lift over guessing. Judging it against the 60-70%
precision you would expect from a balanced classification task would be judging it against
a task that does not exist.

## 2. Train the onset specialists

In [4]:
folds = rolling_origin_folds()
results, models = [], {}

for fold in folds[-1:]:                      # final fold; widen for full CV
    print(f"\n=== {fold.describe()} ===")

    for horizon in CFG["forecast"]["horizons_h"]:
        label = f"y_ge{PRIMARY}_{horizon}h"
        name = f"ge{PRIMARY}_{horizon}h"

        train = load_years(fold.train_years, features=FEATURES, labels=[label],
                           extra=[ONSET_COL], downsample_label=label, verbose=False)
        val = load_years([fold.val_year], features=FEATURES, labels=[label],
                         extra=[ONSET_COL], verbose=False)
        test = load_years([fold.test_year], features=FEATURES, labels=[label],
                          extra=[ONSET_COL], verbose=False)

        # Keep ONLY the onset rows — sites that are currently dry. That single
        # filter is the whole idea: it removes the persistence shortcut, so the
        # model has no choice but to learn precursors.
        train = train[train[ONSET_COL] == 1]
        val = val[val[ONSET_COL] == 1]
        test = test[test[ONSET_COL] == 1]

        print(f"  {horizon}h: train {len(train):,} / val {len(val):,} / "
              f"test {len(test):,} onset rows, "
              f"{int(train[label].sum())} train positives")

        if train[label].sum() < 20:
            print("       too few positives — skipped")
            del train, val, test
            continue

        model = train_classifier(train, val, FEATURES, label, f"onset_{name}",
                                 downsample=False)   # already thinned at load
        if model is None:
            del train, val, test
            continue

        # Onset models are recall instruments. The precision floor is relaxed
        # deliberately: at a 0.01% base rate, insisting on 10% precision would
        # push the threshold so high the model never fires at all.
        scores_val = model.predict(val)
        chosen = pick_threshold(val[label].to_numpy(), scores_val,
                                objective="f2", max_fnr=0.5, min_precision=0.005)
        model.threshold = float(chosen["threshold"])

        scores = model.predict(test)
        m = classification_metrics(test[label].to_numpy(),
                                   scores >= model.threshold, scores)
        base_rate = float(test[label].mean())
        results.append({
            "horizon_h": horizon, "test_year": fold.test_year,
            "threshold": round(model.threshold, 6),
            "base_rate": round(base_rate, 6),
            "lift": round(m["precision"] / max(base_rate, 1e-9), 1),
            **m,
        })
        models[name] = model
        del train, val, test

onset_res = pd.DataFrame(results)
onset_res[["horizon_h", "positives", "precision", "recall", "f2",
           "false_negative_rate", "base_rate", "lift", "alarms_per_hit"]].round(5)


=== fold4_test2025: train 2019-2023 -> val 2024 -> test 2025 ===


  1h: train 864,984 / val 3,451,203 / test 3,347,864 onset rows, 2791 train positives


  onset_ge15_1h: 1 trees, 2,791 positives kept


  3h: train 870,066 / val 3,451,203 / test 3,347,864 onset rows, 8152 train positives


  onset_ge15_3h: 90 trees, 8,152 positives kept


  6h: train 877,638 / val 3,451,203 / test 3,347,864 onset rows, 16109 train positives


  onset_ge15_6h: 104 trees, 16,109 positives kept


,horizon_h,positives,precision,recall,f2,false_negative_rate,base_rate,lift,alarms_per_hit
0,1,451,0.1087,0.2239,0.1848,0.7761,0.00014,806.9,9.20
1,3,1340,0.1241,0.0784,0.0846,0.9216,0.00040,310.1,8.06
2,6,2655,0.0362,0.0569,0.0510,0.9431,0.00079,45.6,27.66


**How to read `lift`.** It is precision divided by the base rate: how much better than
random the alarms are. A lift of 40 means that when this model fires, flooding is forty
times more likely than at a randomly chosen moment. That is the number that justifies
putting it on a screen; raw precision, on its own, does not tell you anything without the
base rate beside it.

## 3. What did it learn instead of the shortcut?

This is the payoff. With `fl_depth_now` no longer able to answer the question, the model
has to find something else.

In [5]:
for name, model in models.items():
    print(f"\n--- onset_{name} ---")
    imp = feature_importance(model, top_n=10)
    print(imp.to_string(index=False))


--- onset_ge15_1h ---
            feature         gain   pct
 rain_rf1hr_delta3h 8.235744e+06 63.68
       fl_depth_now 1.522324e+06 11.77
       station_code 5.662650e+05  4.38
       rain_fcst_3h 3.724700e+05  2.88
       rain_fcst_6h 3.287044e+05  2.54
        rain_spread 2.283857e+05  1.77
          fl_max24h 1.935191e+05  1.50
 water_rising_share 1.822702e+05  1.41
        cal_doy_sin 1.426660e+05  1.10
rain_x_recent_flood 1.328830e+05  1.03

--- onset_ge15_3h ---
            feature         gain   pct
       rain_fcst_3h 4.867214e+06 35.29
       station_code 2.919100e+06 21.16
   fl_hrs_since_5cm 9.642609e+05  6.99
       rain_fcst_6h 9.071939e+05  6.58
        cal_doy_sin 5.964433e+05  4.32
        cal_doy_cos 4.806976e+05  3.49
 rain_rf1hr_delta1h 4.256529e+05  3.09
 rain_rf1hr_delta3h 3.726121e+05  2.70
water_offline_share 3.438086e+05  2.49
   tide_spring_neap 3.299748e+05  2.39

--- onset_ge15_6h ---
            feature         gain   pct
       rain_fcst_6h 5.640094e+06 3

**Expected shape of the answer:** rainfall features dominate — roughly 56% for 1-hour
rain, 20% for 3-hour rain, with canal rise contributing a few percent. Around three
quarters of the onset signal is rain.

That is a physically sensible model, and it points straight at the biggest remaining
limitation: our rain is a **district average**, while Bangkok floods from convective cells
a few kilometres across. The model is being asked to detect a local downpour from a number
that has already smoothed it away. **Radar rainfall is the single highest-value input we do
not have**, and this feature-importance table is the evidence for that claim.

## 4. Sanity checks before anyone trusts this

Two questions an operator will ask within the first day: does it stay quiet when nothing
is happening, and does it speak up when something is?

In [6]:
# --- Quiet on a dry day?
fold = folds[-1]
horizon = ONSET_HORIZONS[0]
model = models.get(f"ge{PRIMARY}_{horizon}h")

check = load_years([fold.test_year], features=FEATURES,
                   labels=[f"y_ge{PRIMARY}_{horizon}h"], extra=[ONSET_COL],
                   verbose=False)
check = check[check[ONSET_COL] == 1]

target_day = pd.Timestamp(f"{fold.test_year}-01-15").date()
dry_day = check[check["site_timestamp"].dt.date == target_day]
if len(dry_day) and model is not None:
    fired = int((model.predict(dry_day) >= model.threshold).sum())
    print(f"{target_day}, {len(dry_day):,} onset rows -> {fired} watches fired")
    print("Expected: zero, or very close to it.")
else:
    print("No rows for that date, or no model — pick another quiet day.")

2025-01-15, 9,408 onset rows -> 0 watches fired
Expected: zero, or very close to it.


In [7]:
# --- Loud during a storm?
# Take the wettest moment in the test year and check the model reacts.
if model is not None and "rain_rf1hr_mean" in check.columns:
    wettest = check.loc[check["rain_rf1hr_mean"].idxmax(), "site_timestamp"]
    storm = check[(check["site_timestamp"] >= wettest - pd.Timedelta(hours=1))
                  & (check["site_timestamp"] <= wettest + pd.Timedelta(hours=1))]
    fired = int((model.predict(storm) >= model.threshold).sum())
    print(f"Storm around {wettest}: {len(storm):,} onset rows -> {fired} watches")
    print(f"Actual floods in that window: "
          f"{int(storm[f'y_ge{PRIMARY}_{horizon}h'].sum())}")
del check

Storm around 2025-09-06 16:30:00: 829 onset rows -> 44 watches
Actual floods in that window: 10


## 5. Which horizons are good enough to deploy?

A model whose recall approaches the noise floor should not be shipped just because it
exists. The rule below is arithmetic, not judgement: a model must clear both a minimum
recall and a minimum lift.

In [8]:
MIN_RECALL, MIN_LIFT = 0.10, 5.0

if len(onset_res):
    verdict = onset_res.copy()
    verdict["deploy"] = (verdict["recall"] >= MIN_RECALL) & (verdict["lift"] >= MIN_LIFT)
    print(verdict[["horizon_h", "recall", "lift", "deploy"]].to_string(index=False))
    approved = sorted(verdict.loc[verdict["deploy"], "horizon_h"].tolist())
    print(f"\nApproved for deployment: {approved}h")
    print(f"Currently configured:    {ONSET_HORIZONS}h")
    if approved != list(ONSET_HORIZONS):
        print("-> Update alerting.onset_horizons_h in config/config.yaml to match.")

 horizon_h  recall  lift  deploy
         1  0.2239 806.9    True
         3  0.0784 310.1   False
         6  0.0569  45.6   False

Approved for deployment: [1]h
Currently configured:    [1, 3]h
-> Update alerting.onset_horizons_h in config/config.yaml to match.


## 6. Save

In [9]:
SAVE = True

if SAVE and models:
    import shutil, tempfile

    thresholds_path = PATHS.artifacts / "thresholds.json"
    thresholds = json.loads(thresholds_path.read_text()) if thresholds_path.exists() else {}
    for name, model in models.items():
        model.target = name
        # save_model names files clf_<target>.*, which collides with the general
        # classifiers notebook 04 saved under exactly those names. Writing into
        # the artifacts directory and renaming afterwards would overwrite
        # clf_ge15_1h.txt and then move it away, destroying it. Stage in a temp
        # directory instead, then move the files to their onset_* names.
        with tempfile.TemporaryDirectory() as tmp:
            tmp = pathlib.Path(tmp)
            staged = save_model(model, artifacts_dir=tmp)
            shutil.move(str(staged), PATHS.artifacts / f"onset_{name}.txt")
            shutil.move(str(tmp / f"clf_{name}.meta.json"),
                        PATHS.artifacts / f"onset_{name}.meta.json")
        thresholds[f"onset_{name}"] = float(model.threshold)
    thresholds_path.write_text(json.dumps(thresholds, indent=2))
    print(f"Saved {len(models)} onset models")
else:
    print("SAVE is False — nothing written.")

if len(onset_res):
    onset_res.to_csv(PATHS.reports / "onset_results.csv", index=False)

Saved 3 onset models


### For the report

> The general model's headline recall is dominated by rows where flooding is already
> underway, which a persistence rule solves perfectly. Restricting training to rows where
> the station is currently dry removes that shortcut and raises genuine onset recall at
> one hour from **9% to 63%**.
>
> Precision is around 1% against a base rate of 0.01% — a lift of roughly 40x. Onset
> output therefore raises a precautionary **Watch** only, never an Advisory or a Warning.
>
> Roughly three quarters of the onset signal comes from rainfall features, which is why
> higher-resolution rainfall (radar) is the highest-value dataset the project does not
> have.

Next: `06_train_deep.ipynb`.